In [1]:
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark context
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [3]:
# inspect spark session
spark

In [4]:
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [5]:
base_aug_df = spark.read.parquet("../Data/Cleaned_data/biweekly_augmented.parquet")

In [6]:
base_aug_df.show(5)

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+-------------------+-------+---------+
|ticker|biweekly_index|biweekly_period|            longname|industry|benchmark|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|benchmark_period_return|                beta|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volume| volume_volatility|avg_volume_up_days|avg_volume_down_days|gap_count|        avg_gap_size|            max_gap|up_days|down_days|
+---

In [7]:
import glob
import os

In [8]:
import pandas as pd

In [9]:
df = pd.read_csv("./Data/Dependent_data/AYA.TO_announcements.csv")

In [10]:
base_path = "./Data/Dependent_data/"

# Get every file ending with _announcements.csv
csv_paths = glob.glob(os.path.join(base_path, "*_announcements.csv"))

dep_df = (
    spark.read
        .option("header", True)
        .csv(csv_paths)
)

In [11]:
dep_df.show()

+--------------------+------+----------+--------------------+
|          stock_name|ticker| date-time|                type|
+--------------------+------+----------+--------------------+
|Aya Gold & Silver...|AYA.TO|2024-01-11| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-02-06|financing_announc...|
|Aya Gold & Silver...|AYA.TO|2024-02-14|closing_announcement|
|Aya Gold & Silver...|AYA.TO|2024-02-20|closing_announcement|
|Aya Gold & Silver...|AYA.TO|2024-03-19|property_acquisition|
|Aya Gold & Silver...|AYA.TO|2024-04-11| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-05-13|property_acquisition|
|Aya Gold & Silver...|AYA.TO|2024-06-24|closing_announcement|
|Aya Gold & Silver...|AYA.TO|2024-07-02| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-07-23| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-09-04| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-09-12|property_acquisition|
|Aya Gold & Silver...|AYA.TO|2024-09-16| exploration_results|
|Aya Gol

In [12]:
base_dates = (
    base_aug_df.groupBy("ticker")
      .agg(
          F.min("period_start").alias("min_period_start"),
          F.max("period_end").alias("max_period_end")
      )
)

base_dates.show()

+-------+----------------+--------------+
| ticker|min_period_start|max_period_end|
+-------+----------------+--------------+
| ARG.TO|      2024-01-15|    2025-08-29|
|  ESK.V|      2024-07-18|    2025-08-29|
|  FNM.V|      2024-04-29|    2025-08-29|
|FURY.TO|      2024-07-18|    2025-08-29|
| FOM.TO|      2024-10-15|    2025-10-10|
| FINX.V|      2025-06-18|    2025-10-15|
| ELO.TO|      2024-01-15|    2025-08-29|
|  EDG.V|      2024-07-22|    2025-08-29|
| FVL.TO|      2024-01-18|    2025-08-29|
|  CGC.V|      2024-10-10|    2025-10-09|
|  CKG.V|      2024-10-10|    2025-10-09|
|   DV.V|      2024-09-17|    2025-09-17|
| AYA.TO|      2024-10-15|    2025-10-10|
|  DMX.V|      2024-01-18|    2025-10-15|
| FDY.TO|      2024-10-15|    2025-10-15|
|  FWZ.V|      2024-07-25|    2025-08-29|
|  FAS.V|      2024-10-16|    2025-10-15|
|  CG.TO|      2024-10-10|    2025-10-09|
|    EQX|      2024-07-02|    2025-08-29|
+-------+----------------+--------------+



In [13]:
start = base_dates.select(F.min("min_period_start")).collect()[0][0]
end = base_dates.select(F.max("max_period_end")).collect()[0][0]
print("Start date: ", start, "\nEnd date: ", end)

Start date:  2024-01-15 
End date:  2025-10-15


In [14]:
start_dep = dep_df.select(F.min("date-time")).collect()[0][0]
end_dep = dep_df.select(F.max("date-time")).collect()[0][0]
print("Start date: ", start_dep, "\nEnd date: ", end_dep)

Start date:  2024-01-08 
End date:  2026-05-10


In [15]:
dep_df = dep_df.filter(
    (F.col("date-time") >= start) &
    (F.col("date-time") <= end)
)
dep_df.show()

+--------------------+------+----------+--------------------+
|          stock_name|ticker| date-time|                type|
+--------------------+------+----------+--------------------+
|Aya Gold & Silver...|AYA.TO|2024-02-06|financing_announc...|
|Aya Gold & Silver...|AYA.TO|2024-02-14|closing_announcement|
|Aya Gold & Silver...|AYA.TO|2024-02-20|closing_announcement|
|Aya Gold & Silver...|AYA.TO|2024-03-19|property_acquisition|
|Aya Gold & Silver...|AYA.TO|2024-04-11| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-05-13|property_acquisition|
|Aya Gold & Silver...|AYA.TO|2024-06-24|closing_announcement|
|Aya Gold & Silver...|AYA.TO|2024-07-02| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-07-23| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-09-04| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-09-12|property_acquisition|
|Aya Gold & Silver...|AYA.TO|2024-09-16| exploration_results|
|Aya Gold & Silver...|AYA.TO|2024-11-19| exploration_results|
|Aya Gol

adding biweekly index to announcements

In [16]:
def biweekly_announcements(df):

    # --- PREP ---
    df = df.withColumnRenamed("date-time", "event_date")

    min_date = df.agg(F.min("event_date")).collect()[0][0]

    df = df.withColumn(
        "days_since_start",
        F.datediff(F.col("event_date"), F.lit(min_date))
    )

    df = df.withColumn(
        "biweekly_index",
        (F.col("days_since_start") / F.lit(14)).cast("int")
    )

    df = df.withColumn(
        "biweekly_period",
        F.concat(F.lit("P"), F.col("biweekly_index"))
    )

    df = df.withColumnRenamed("stock_name", "longname")

    # --- ROW-LEVEL FLAGS ---
    df = (
        df
        .withColumn("is_financing", 
                    F.when(F.col("type") == "financing_announcement", 1).otherwise(0))
        .withColumn("is_property",  
                    F.when(F.col("type") == "property_acquisition", 1).otherwise(0))
        .withColumn("is_exploration", 
                    F.when(F.col("type") == "exploration_results", 1).otherwise(0))
    )


    # --- AGGREGATE PER BIWEEK ---
    grouped = (
        df.groupBy("ticker", "longname", "biweekly_index", "biweekly_period")
          .agg(
              F.max("is_financing").alias("has_financing"),
              F.max("is_property").alias("has_property_acquisition"),
              F.max("is_exploration").alias("has_exploration_results")
          )
    )

    # --- FORWARD-LOOKING WINDOW ---
    w = Window.partitionBy("ticker").orderBy("biweekly_index")

    biweekly = (
        grouped
        .withColumn("financing_next_period",
                    F.lead("has_financing", 1).over(w))
        .withColumn("property_next_period",
                    F.lead("has_property_acquisition", 1).over(w))
        .withColumn("exploration_next_period",
                    F.lead("has_exploration_results", 1).over(w))
        .fillna(0, subset=[
            "financing_next_period",
            "property_next_period",
            "exploration_next_period"
        ])
    )

    return biweekly

In [17]:
biweekly_dep = biweekly_announcements(dep_df)
biweekly_dep.show()

+------+--------------------+--------------+---------------+-------------+------------------------+-----------------------+---------------------+--------------------+-----------------------+
|ticker|            longname|biweekly_index|biweekly_period|has_financing|has_property_acquisition|has_exploration_results|financing_next_period|property_next_period|exploration_next_period|
+------+--------------------+--------------+---------------+-------------+------------------------+-----------------------+---------------------+--------------------+-----------------------+
|AYA.TO|Aya Gold & Silver...|             1|             P1|            1|                       0|                      0|                    0|                   0|                      0|
|AYA.TO|Aya Gold & Silver...|             2|             P2|            0|                       0|                      0|                    0|                   1|                      0|
|AYA.TO|Aya Gold & Silver...|             4| 

In [ ]:
dep_sel = (
    biweekly_dep
    .select(
        "ticker",
        "biweekly_index",
        "has_financing"
        "financing_next_period",
        "has_property_acquisition",
        "has_exploration_results"
    )
)

# 3) Left join onto base_df on ticker + biweekly_index
joined = (
    base_aug_df
    .join(dep_sel, on=["ticker", "biweekly_index"], how="left")
)

# 4) Replace nulls in the dummy columns with 0
joined = joined.fillna(
    {
        "financing_next_period": 0,
        "has_property_acquisition": 0,
        "has_exploration_results": 0,
    }
)

# (Optional) make sure they’re integers, not doubles
joined = (
    joined
    .withColumn("financing_next_period", F.col("financing_next_period").cast("int"))
    .withColumn("has_property_acquisition", F.col("has_property_acquisition").cast("int"))
    .withColumn("has_exploration_results", F.col("has_exploration_results").cast("int"))
)

In [19]:
joined.filter(F.col("ticker") == "AYA.TO").show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+-------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+
|ticker|biweekly_index|biweekly_period|            longname|industry|benchmark|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|benchmark_period_return|                beta|   price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volume| volume_volatility|avg_volume_up_days|avg_volume_down_days|gap_

In [20]:
joined.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable.parquet")